# Advanced Linear Regression - Assumptions, Scaling & Regularization

## 🎯 Goals
1. Check all Linear Regression assumptions
2. Learn about feature scaling
3. Apply regularization (Ridge, Lasso, Elastic Net)
4. Handle real-world challenges

## 📚 What You'll Learn
- How to check if assumptions are met
- When and how to scale features
- How to prevent overfitting with regularization
- How to choose the best model

## Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Set random seed
np.random.seed(42)

# Set plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("✅ Libraries imported successfully!")

## Step 2: Create Sample Data

Generate realistic house price data with some noise.

In [ ]:
# Generate data
n_samples = 100

size = np.random.uniform(1000, 4000, n_samples)
bedrooms = np.random.randint(1, 6, n_samples)
age = np.random.uniform(0, 50, n_samples)

# True relationship with some noise
price = 50 + 0.2*size + 30*bedrooms - 1*age + np.random.normal(0, 20, n_samples)

# Create DataFrame
df = pd.DataFrame({
    'size': size,
    'bedrooms': bedrooms,
    'age': age,
    'price': price
})

print("📊 Dataset Created")
print("=" * 50)
print(f"Samples: {len(df)}")
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nStatistics:")
print(df.describe())

## Step 3: Train Initial Model

Train a basic model to check assumptions.

In [ ]:
# Prepare data
X = df[['size', 'bedrooms', 'age']]
y = df['price']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
model = LinearRegression()
model.fit(X_train, y_train)

# Get predictions and residuals
y_pred = model.predict(X)
residuals = y - y_pred

print("🤖 Model Trained!")
print("=" * 50)
print(f"Intercept: ${model.intercept_:.2f}k")
print(f"\nCoefficients:")
for feature, coef in zip(X.columns, model.coef_):
    print(f"  {feature:10s}: ${coef:8.4f}k")

## ASSUMPTION CHECKS

---

## Assumption 1: Linearity

**Check:** Plot each feature vs target. Should see straight line patterns.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, col in enumerate(['size', 'bedrooms', 'age']):
    axes[i].scatter(X[col], y, alpha=0.5)
    axes[i].set_xlabel(col, fontsize=11)
    axes[i].set_ylabel('price', fontsize=11)
    axes[i].set_title(f'Price vs {col}', fontweight='bold')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Linearity Check:")
print("   Look for: Straight line patterns")
print("   Watch out for: Curved patterns (would need polynomial features)")

## Assumption 2: Homoscedasticity (Constant Variance)

**Check:** Plot residuals vs predicted values. Spread should be constant.

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(y_pred, residuals, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--', linewidth=2)
plt.xlabel('Predicted Values', fontsize=12)
plt.ylabel('Residuals', fontsize=12)
plt.title('Residual Plot - Check for Homoscedasticity', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✅ Homoscedasticity Check:")
print("   Good: Even spread (rectangle shape)")
print("   Bad: Funnel shape (variance increases)")
print("\n   Current: Points are evenly scattered ✅")

## Assumption 3: Normality of Residuals

**Check:** Histogram and Q-Q plot of residuals.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(residuals, bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Residuals', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].set_title('Distribution of Residuals', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Q-Q Plot
stats.probplot(residuals, dist="norm", plot=axes[1])
axes[1].set_title('Q-Q Plot', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Normality Check:")
print("   Histogram: Should look like bell curve")
print("   Q-Q Plot: Points should be on diagonal line")
print("\n   Current: Residuals appear normally distributed ✅")

## Assumption 4: No Multicollinearity

**Check:** Correlation matrix and VIF (Variance Inflation Factor).

In [ ]:
# Correlation matrix
corr_matrix = X.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, fmt='.2f')
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("📊 Correlation Matrix:")
print(corr_matrix)

# Calculate VIF
vif_data = pd.DataFrame()
vif_data["Feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) 
                   for i in range(len(X.columns))]

print("\n📊 Variance Inflation Factor (VIF):")
print(vif_data)

print("\n✅ Multicollinearity Check:")
print("   VIF < 5: Low multicollinearity ✅")
print("   VIF 5-10: Moderate multicollinearity ⚠️")
print("   VIF > 10: High multicollinearity ❌")
print("\n   Current: All VIF values are low ✅")

## FEATURE SCALING

---

## Why Scale Features?

Features are on different scales:
- Size: 1000-4000 (large numbers)
- Bedrooms: 1-5 (small numbers)
- Age: 0-50 (medium numbers)

This makes coefficients hard to compare!

In [ ]:
print("📊 Feature Scales (Before Scaling)")
print("=" * 50)
print(X.describe())

print("\n📋 Unscaled Coefficients:")
for feature, coef in zip(X.columns, model.coef_):
    print(f"  {feature:10s}: {coef:10.4f}")

print("\n⚠️  Hard to compare! Different scales make comparison misleading.")

## Apply StandardScaler (Z-score Normalization)

Transforms features to have mean=0 and std=1.

In [ ]:
# Create scaler
scaler = StandardScaler()

# Fit and transform
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model on scaled data
model_scaled = LinearRegression()
model_scaled.fit(X_train_scaled, y_train)

print("✅ Features Scaled!")
print("\n📋 Scaled Coefficients:")
for feature, coef in zip(X.columns, model_scaled.coef_):
    print(f"  {feature:10s}: {coef:10.2f}")

print("\n💡 Now coefficients are directly comparable!")
print("   Larger absolute value = more important feature")

# Find most important
most_important_idx = np.argmax(np.abs(model_scaled.coef_))
most_important = X.columns[most_important_idx]
print(f"\n🏆 Most important feature: {most_important}")

## REGULARIZATION

---

## Ridge Regression (L2 Regularization)

Shrinks coefficients to prevent overfitting.

In [ ]:
# Try different alpha values
alphas = [0.01, 0.1, 1, 10, 100]
ridge_results = []

print("🔵 Ridge Regression - Testing Different Alphas")
print("=" * 70)

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_scaled, y_train)
    
    train_score = ridge.score(X_train_scaled, y_train)
    test_score = ridge.score(X_test_scaled, y_test)
    
    ridge_results.append({
        'alpha': alpha,
        'train_r2': train_score,
        'test_r2': test_score,
        'coefficients': ridge.coef_.copy()
    })
    
    print(f"\nAlpha = {alpha:6.2f}")
    print(f"  Train R²: {train_score:.3f}")
    print(f"  Test R²:  {test_score:.3f}")
    print(f"  Coefs: {ridge.coef_.round(2)}")

# Plot alpha vs performance
plt.figure(figsize=(10, 6))
plt.semilogx(alphas, [r['test_r2'] for r in ridge_results], 
             'o-', linewidth=2, markersize=8, label='Test R²')
plt.semilogx(alphas, [r['train_r2'] for r in ridge_results], 
             's--', linewidth=2, markersize=8, label='Train R²', alpha=0.7)
plt.xlabel('Alpha (Regularization Strength)', fontsize=12)
plt.ylabel('R² Score', fontsize=12)
plt.title('Ridge Regression: Alpha vs Performance', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_ridge = max(ridge_results, key=lambda x: x['test_r2'])
print(f"\n✅ Best alpha: {best_ridge['alpha']} (Test R² = {best_ridge['test_r2']:.3f})")

## Lasso Regression (L1 Regularization)

Shrinks coefficients AND can eliminate features (set to 0).

In [ ]:
print("🔴 Lasso Regression - Feature Selection")
print("=" * 70)

# Train Lasso
lasso = Lasso(alpha=1)
lasso.fit(X_train_scaled, y_train)

print("\nLasso Coefficients:")
for feature, coef in zip(X.columns, lasso.coef_):
    status = "✅ KEPT" if coef != 0 else "❌ ELIMINATED"
    print(f"  {feature:10s}: {coef:10.4f}  {status}")

n_features_kept = np.sum(lasso.coef_ != 0)
print(f"\nFeatures kept: {n_features_kept}/{len(X.columns)}")

test_score = lasso.score(X_test_scaled, y_test)
print(f"Test R²: {test_score:.3f}")

print("\n💡 Lasso automatically selected the most important features!")

## Elastic Net (Ridge + Lasso)

Combines benefits of both!

In [ ]:
print("🟣 Elastic Net - Best of Both Worlds")
print("=" * 70)

# Train Elastic Net
elastic = ElasticNet(alpha=1, l1_ratio=0.5)
elastic.fit(X_train_scaled, y_train)

print("\nElastic Net Coefficients:")
for feature, coef in zip(X.columns, elastic.coef_):
    status = "✅ KEPT" if abs(coef) > 0.01 else "❌ ELIMINATED"
    print(f"  {feature:10s}: {coef:10.4f}  {status}")

test_score = elastic.score(X_test_scaled, y_test)
print(f"\nTest R²: {test_score:.3f}")

## Compare All Models

In [ ]:
# Train all models
models = {
    'Linear (unscaled)': LinearRegression().fit(X_train, y_train),
    'Linear (scaled)': LinearRegression().fit(X_train_scaled, y_train),
    'Ridge': Ridge(alpha=best_ridge['alpha']).fit(X_train_scaled, y_train),
    'Lasso': lasso,
    'Elastic Net': elastic
}

# Evaluate
results = []
for name, model in models.items():
    if 'unscaled' in name:
        train_score = model.score(X_train, y_train)
        test_score = model.score(X_test, y_test)
    else:
        train_score = model.score(X_train_scaled, y_train)
        test_score = model.score(X_test_scaled, y_test)
    
    results.append({
        'Model': name,
        'Train R²': train_score,
        'Test R²': test_score,
        'Difference': abs(train_score - test_score)
    })

results_df = pd.DataFrame(results)

print("📊 Model Comparison")
print("=" * 70)
print(results_df.to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(results_df))
width = 0.35

ax.bar(x - width/2, results_df['Train R²'], width, label='Train R²', alpha=0.8)
ax.bar(x + width/2, results_df['Test R²'], width, label='Test R²', alpha=0.8)

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('R² Score', fontsize=12)
ax.set_title('Model Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(results_df['Model'], rotation=45, ha='right')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\n💡 Look for:")
print("   - High test R² (good performance)")
print("   - Small train-test gap (good generalization)")

best_model = results_df.loc[results_df['Test R²'].idxmax()]
print(f"\n🏆 Best model: {best_model['Model']} (Test R² = {best_model['Test R²']:.3f})")

## 🎓 Summary

### Assumptions Checked:
1. ✅ **Linearity**: Relationships appear linear
2. ✅ **Homoscedasticity**: Constant variance of residuals
3. ✅ **Normality**: Residuals are normally distributed
4. ✅ **No Multicollinearity**: Low VIF values

### Scaling:
- **StandardScaler**: Makes coefficients comparable
- **Important**: Necessary for regularization
- **Interpretation**: Changes but relationships stay same

### Regularization:
- **Ridge**: Shrinks all coefficients, keeps all features
- **Lasso**: Shrinks + eliminates features (feature selection)
- **Elastic Net**: Combines both approaches

### Key Takeaways:
1. Always check assumptions before trusting results
2. Scale features when they're on different scales
3. Use regularization to prevent overfitting
4. Compare multiple models to find the best
5. Look for small train-test gap (good generalization)

**Happy Learning! 🚀**